# FlyRank ML Capstone
This notebook reproduces the data extraction and modeling for the capstone paper.

In [ ]:
import duckdb
import pandas as pd
import os

HF_TOKEN = 'YOUR_HF_TOKEN_HERE'
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

In [ ]:
features = con.sql("""
        WITH bounds AS (
            SELECT MAX(report_date) AS end_d FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
        ),
        windowed AS (
            SELECT f.client_hash_id, f.content_hash_id,
                   SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
                   SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
                   SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 15 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
                   SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 15 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
                   AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 15 DAY THEN f.gsc_avg_position END)       AS pos_last30,
                   AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 15 DAY THEN f.gsc_avg_position END)       AS pos_prev30
            FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet') f, bounds b
            WHERE f.report_date > b.end_d - INTERVAL 30 DAY
            GROUP BY 1, 2
            HAVING imp_prev30 >= 100
        )
        SELECT * FROM windowed
    """).df()
qsignals = con.sql("""
        SELECT content_hash_id,
               ANY_VALUE(content_visible_query_count)     AS visible_queries,
               ANY_VALUE(rare_impressions_share)          AS rare_share,
               ANY_VALUE(anonymized_impressions_share)    AS anon_share,
               MAX(impressions_90d)                       AS top_query_impressions,
               SUM(impressions_90d)                       AS kept_impressions
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')
        GROUP BY content_hash_id
    """).df()

In [ ]:
data = features.merge(qsignals, on='content_hash_id', how='left')
data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
# ... continuing with standard sklearn workflow ...